# `POST /validate-config` — assignment examples

Manual checks against the running server. Each cell sends one of the three input/output examples from `specs/assignment.md` to `POST /validate-config` and prints the prettified JSON response.

**Prereqs**

- Server running locally: `npm run dev:server` (or `npm run dev`).
- `OPENAI_API_KEY` set in `server/.env` (otherwise the endpoint returns `502`).
- Python `requests` available: `pip install requests`.

In [1]:
import json
import requests

BASE_URL = "http://localhost:3000"
ENDPOINT = f"{BASE_URL}/validate-config"
MODEL = "gpt-5"  # e.g. "gpt-4o-mini" or "gpt-4o"; None uses the server default

def call_validate(config: dict, model: str | None = MODEL) -> None:
    """POST `config` to /validate-config and pretty-print the response."""
    params = {"model": model} if model else None
    print("Request:")
    print(json.dumps(config, indent=2))
    print()
    response = requests.post(ENDPOINT, json=config, params=params, timeout=60)
    print(f"HTTP {response.status_code}")
    try:
        body = response.json()
        print(json.dumps(body, indent=2, ensure_ascii=False))
    except ValueError:
        print(response.text)

## Example 1 — reward too high for an easy level

Expected pattern: schema is valid; the LLM should flag a `reward_vs_difficulty` mismatch (5000 reward on `easy`).

In [2]:
call_validate({
    "level": 12,
    "time_limit": 60,
    "reward": 5000,
    "difficulty": "easy"
})

Request:
{
  "level": 12,
  "time_limit": 60,
  "reward": 5000,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Early-game level 12 is marked easy with a very generous 60s timer, but it awards 5000 currency, which aligns with hard-tier rewards and drives a high reward-per-second. The mismatch between declared difficulty and payout creates a strong farming incentive and economy inflation risk.",
    "suggested_actions": [
      "Reduce reward to 100–500 for easy difficulty, or reclassify as hard and cut the timer to 10–30s if keeping a 5000 reward.",
      "Lower reward or tighten the timer so reward-per-second fits easy expectations (roughly 2–17 per second).",
      "Lower the reward to the easy tier (100–500) to avoid early-game farming and currency inflation."
    ],
    "confidence": 0.92
  }
}


## Example 2 — time limit too tight for a hard level

Expected pattern: schema is valid; the LLM should flag `time_vs_difficulty` (10s on `hard`) and likely `frustration_risk`.

In [3]:
call_validate({
    "level": 5,
    "time_limit": 10,
    "reward": 500,
    "difficulty": "hard"
})

Request:
{
  "level": 5,
  "time_limit": 10,
  "reward": 500,
  "difficulty": "hard"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 5 is marked hard with a very tight 10s timer but only awards 500 currency. The timer fits a hard experience, yet the reward aligns with easy/low‑medium ranges. Given this is an early level in a 150‑level game, labeling it hard breaks expected progression, and the combo of tight time and low payout risks feeling punishing early on.",
    "suggested_actions": [
      "Increase reward to roughly 2000–5000 to match hard difficulty.",
      "Downgrade the difficulty to easy/medium for level 5 or move this configuration to a much later level (e.g., 120+).",
      "If kept early, relax the timer to around 20–30s or substantially raise the reward so the effort feels worthwhile."
    ],
    "confidence": 0.89
  }
}


## Example 3 — reasonable starting level (expect empty findings)

Expected pattern: schema is valid; the LLM should return an empty `findings` array, so `suggested_actions` is `["No action needed"]` and `confidence` is the model's `verdict_confidence`.

In [4]:
call_validate({
    "level": 1,
    "time_limit": 120,
    "reward": 100,
    "difficulty": "easy"
})

Request:
{
  "level": 1,
  "time_limit": 120,
  "reward": 100,
  "difficulty": "easy"
}

HTTP 200
{
  "schema_validation": {
    "valid": true,
    "errors": []
  },
  "llm_feedback": {
    "analysis": "Level 1 is marked easy with a very generous 120s timer and a minimal reward of 100. Difficulty vs. level placement is appropriate and the reward band matches easy, but the long timer makes the reward-per-second unusually low for an onboarding level. This may make the first clear feel grindy or slow-paced despite being easy.",
    "suggested_actions": [
      "Either reduce the time_limit to around 45–60s or increase the reward to roughly 200–300 to bring reward-per-second in line with easy levels."
    ],
    "confidence": 0.7
  }
}


## Bonus — schema-validation failure (expect HTTP 400, no LLM call)

Sends a malformed body to confirm the Zod gate rejects it before the LLM is called.

In [5]:
call_validate({"level": "oops"})

Request:
{
  "level": "oops"
}

HTTP 400
{
  "schema_validation": {
    "valid": false,
    "errors": [
      {
        "path": "level",
        "message": "Expected number, received string"
      },
      {
        "path": "time_limit",
        "message": "Required"
      },
      {
        "path": "reward",
        "message": "Required"
      },
      {
        "path": "difficulty",
        "message": "Required"
      }
    ]
  }
}
